# Notebook 02 — SAE Training
Trains one SAE per layer (layers 4 and 12). Logs MSE, L0, dead features every 1k steps.
Target L0: 5–50 active features per token. Adjust alpha if outside this range.

In [ ]:
import sys, os
sys.path.insert(0, '..')
import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from src.sae import SAE

DEVICE    = 'cuda'
D_INPUT   = 768
D_HIDDEN  = 3072
ALPHA     = 1e-3   # increase to 5e-3 if L0 > 100; decrease to 1e-4 if L0 < 2
LR        = 1e-4
BATCH     = 2048
N_STEPS   = 100_000
LOG_EVERY = 1_000
os.makedirs('../checkpoints', exist_ok=True)

In [ ]:
def train_sae(acts_path: str, save_path: str, label: str) -> dict:
    acts = torch.load(acts_path, weights_only=False).to(DEVICE)
    loader = DataLoader(TensorDataset(acts), batch_size=BATCH, shuffle=True, drop_last=True)

    sae = SAE(d_input=D_INPUT, d_hidden=D_HIDDEN, alpha=ALPHA).to(DEVICE)
    opt = torch.optim.Adam(sae.parameters(), lr=LR)

    log = {k: [] for k in ('step', 'total', 'mse', 'l1', 'l0', 'dead_pct')}
    step = 0

    while step < N_STEPS:
        for (x,) in loader:
            if step >= N_STEPS:
                break
            f, x_hat = sae(x)
            total, mse, l1 = sae.loss(x, f, x_hat)
            opt.zero_grad(); total.backward(); opt.step()
            sae.normalize_decoder()

            if step % LOG_EVERY == 0:
                with torch.no_grad():
                    l0      = (f > 0).float().sum(dim=1).mean().item()
                    dead_pct = ((f > 0).float().sum(dim=0) == 0).float().mean().item() * 100
                log['step'].append(step)
                log['total'].append(total.item())
                log['mse'].append(mse.item())
                log['l1'].append(l1.item())
                log['l0'].append(l0)
                log['dead_pct'].append(dead_pct)
                print(f'[{label}] step={step:6d} | mse={mse.item():.4f} | '
                      f'l0={l0:.1f} | dead={dead_pct:.1f}%')
            step += 1

    torch.save({'state_dict': sae.state_dict(), 'log': log}, save_path)
    print(f'Saved {save_path}')
    return log

In [ ]:
log4 = train_sae('../data/layer4_activations.pt', '../checkpoints/sae_layer4.pt', 'layer4')

In [ ]:
log12 = train_sae('../data/layer12_activations.pt', '../checkpoints/sae_layer12.pt', 'layer12')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for log, label, color in [(log4, 'Layer 4', 'steelblue'), (log12, 'Layer 12', 'crimson')]:
    s = log['step']
    axes[0,0].plot(s, log['mse'],      label=label, color=color)
    axes[0,1].plot(s, log['l0'],       label=label, color=color)
    axes[1,0].plot(s, log['dead_pct'], label=label, color=color)
    axes[1,1].plot(s, log['total'],    label=label, color=color)
for ax, title in zip(axes.flat, ['MSE', 'L0 (avg active features)', 'Dead Feature %', 'Total Loss']):
    ax.set_title(title); ax.legend()
plt.tight_layout()
plt.savefig('../checkpoints/training_curves.png', dpi=150)
plt.show()